<a href="https://colab.research.google.com/github/zebsoftware/Machine-Learning/blob/main/Malwaredetectionsystem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import json
import tarfile
import pandas as pd

In [ ]:
archive = "/content/drive/MyDrive/ember_dataset_2018_2.tar.bz2"

with tarfile.open(archive, "r:bz2") as tar:
    tar.extractall("/content/ember")

print("Extraction completed!")

/tmp/ipykernel_34620/2675741230.py:4: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall("/content/ember")


EOFError: Compressed file ended before the end-of-stream marker was reached

In [ ]:
archive = "/content/drive/MyDrive/ember_dataset_2018_2.tar.bz2"

with tarfile.open(archive, "r:bz2") as tar:
    tar.extractall("/content/ember")

print("Extraction completed!")

In [ ]:
folder = "/content/ember/ember2018"

for file in sorted(os.listdir(folder)):
    print(file)

In [ ]:
def extract_features(sample):

    # ----------------------------
    # Access different sections
    # ----------------------------
    general = sample["general"]
    strings = sample["strings"]
    section = sample["section"]
    imports = sample["imports"]
    datadirectories = sample["datadirectories"]

    sections = section["sections"]

    # ----------------------------
    # Section Features
    # ----------------------------
    num_sections = len(sections)

    entropies = [s["entropy"] for s in sections]
    sizes = [s["size"] for s in sections]

    avg_entropy = sum(entropies) / num_sections
    max_entropy = max(entropies)

    avg_size = sum(sizes) / num_sections
    max_size = max(sizes)

    entry_is_text = 1 if section["entry"] == ".text" else 0

    # ----------------------------
    # Import Features
    # ----------------------------
    num_dlls = len(imports)

    total_imports = sum(len(funcs) for funcs in imports.values())

    suspicious_list = {
        "VirtualAlloc",
        "WriteProcessMemory",
        "CreateRemoteThread",
        "LoadLibraryA",
        "GetProcAddress",
        "CreateProcessA",
        "ShellExecuteA",
        "DeleteFileA",
        "RegSetValueExA"
    }

    suspicious_count = 0

    for funcs in imports.values():
        for func in funcs:
            if func in suspicious_list:
                suspicious_count += 1

    # ----------------------------
    # Data Directory Features
    # ----------------------------
    num_directories = len(datadirectories)

    total_directory_size = sum(
        d["size"] for d in datadirectories
    )

    import_directory_size = datadirectories[1]["size"]
    resource_directory_size = datadirectories[2]["size"]

    # ----------------------------
    # Final Feature Vector
    # ----------------------------
    features = [

        # ===== General Features =====
        general["size"],
        general["vsize"],
        general["imports"],
        general["exports"],
        general["has_debug"],
        general["has_relocations"],
        general["has_resources"],
        general["has_signature"],
        general["has_tls"],
        general["symbols"],

        # ===== String Features =====
        strings["numstrings"],
        strings["avlength"],
        strings["entropy"],
        strings["printables"],
        strings["paths"],
        strings["registry"],
        strings["urls"],

        # ===== Section Features =====
        num_sections,
        avg_entropy,
        max_entropy,
        avg_size,
        max_size,
        entry_is_text,

        # ===== Import Features =====
        num_dlls,
        total_imports,
        suspicious_count,

        # ===== Data Directory Features =====
        num_directories,
        total_directory_size,
        import_directory_size,
        resource_directory_size
    ]

    return features

In [ ]:
feature_names = [

    # ===== General Features =====
    "size",
    "vsize",
    "imports",
    "exports",
    "has_debug",
    "has_relocations",
    "has_resources",
    "has_signature",
    "has_tls",
    "symbols",

    # ===== String Features =====
    "numstrings",
    "average_string_length",
    "string_entropy",
    "printable_characters",
    "num_paths",
    "num_registry",
    "num_urls",

    # ===== Section Features =====
    "num_sections",
    "avg_section_entropy",
    "max_section_entropy",
    "avg_section_size",
    "largest_section_size",
    "entry_is_text",

    # ===== Import Features =====
    "num_dlls",
    "total_imports",
    "suspicious_api_count",

    # ===== Data Directory Features =====
    "num_directories",
    "total_directory_size",
    "import_directory_size",
    "resource_directory_size"
]

In [ ]:
import json

file_path = "/content/ember/ember2018/train_features_0.jsonl"

with open(file_path, "r") as f:
    sample = json.loads(f.readline())

features = extract_features(sample)

print("Number of Features:", len(features))
print()

for name, value in zip(feature_names, features):
    print(f"{name:25}: {value}")